# Search Engine Demo

This notebook demonstrates the functionality of our single-file search engine prototype.

## Features Demonstrated:
- **BM25 keyword relevance** with field boosts
- **Synonym expansion** (cv → resume)
- **Facet-based boosting** (size, style, color)
- **Popularity signals** with normalized bonuses
- **MMR diversity re-ranking** using optimized sparse TF-IDF

The search engine operates on 10 mock Canva-like design assets.

In [1]:
# Import the search engine and all its components
import sys
sys.path.append('../src')

from search_engine import (
    CONFIG, DOCS, 
    search, create_search_context,
    tokenize, expand_with_synonyms, detect_facets,
    calculate_bm25_scores, apply_facet_boosts, apply_popularity_bonus,
    mmr_select, cosine_similarity_optimized
)

import pandas as pd
import json
from pprint import pprint

## 1. Setup and Configuration

Let's first examine our configuration and mock dataset.

In [2]:
# Display configuration
print("🔧 SEARCH ENGINE CONFIGURATION")
print("=" * 40)
for key, value in CONFIG.items():
    print(f"{key}: {value}")

print(f"\n📚 DATASET: {len(DOCS)} mock documents")

🔧 SEARCH ENGINE CONFIGURATION
k1: 1.2
b: 0.75
field_boosts: {'title': 2.0, 'tags': 1.5, 'desc': 1.0}
synonyms: {'cv': ['resume'], 'photo': ['image', 'picture'], 'logo': ['brandmark'], 'ppt': ['presentation', 'slides'], 'a4': ['letter']}
syn_weight: 0.7
facets: {'size': ['A4', 'A3', 'square', 'instagram'], 'style': ['minimal', 'vintage', 'modern'], 'color': ['gold', 'blue', 'green', 'black', 'white']}
facet_boost: 2.0
pop_weight: 0.1
mmr_lambda: 0.7

📚 DATASET: 10 mock documents


In [3]:
# Display our mock dataset as a DataFrame for better visualization
docs_df = pd.DataFrame(DOCS)
print("📄 MOCK DATASET OVERVIEW")
print("=" * 40)
display(docs_df[['id', 'title', 'tags', 'popularity', 'size', 'style', 'color']])

📄 MOCK DATASET OVERVIEW


,id,title,tags,popularity,size,style,color
0,d1,Minimal A4 resume template,"[resume, template, a4, minimal]",120,A4,minimal,white
1,d2,Vintage poster design A3,"[poster, vintage, a3]",90,A3,vintage,black
2,d3,Modern business presentation slides,"[presentation, slides, modern]",200,square,modern,blue
3,d4,Gold logo brandmark pack,"[logo, brandmark, gold]",160,square,modern,gold
4,d5,Instagram post template minimal,"[instagram, template, minimal]",300,instagram,minimal,white
5,d6,Professional CV resume pack,"[cv, resume, professional]",80,A4,modern,black
6,d7,Travel photo collage template,"[photo, image, collage, template]",140,square,minimal,white
7,d8,Business card minimal black,"[card, business, minimal, black]",60,A4,minimal,black
8,d9,Event flyer modern blue,"[flyer, event, modern, blue]",110,A4,modern,blue
9,d10,Photography portfolio presentation,"[photography, portfolio, presentation]",95,square,modern,white


## 2. Initialize Search Context

The search context precomputes all the data structures we need for fast searching.

In [4]:
# Create the search context (this happens once)
print("🏗️  Building search context...")
ctx = create_search_context(DOCS, CONFIG)

print("✅ Search context created!")
print(f"   - BM25 models for fields: {list(ctx['bm25'].keys())}")
print(f"   - TF-IDF matrix shape: {ctx['tfidf_matrix'].shape}")
print(f"   - Document norms precomputed: {len(ctx['doc_norms'])}")
print(f"   - Popularity scores normalized: {len(ctx['popularity'])}")
print(f"   - Facets indexed: {list(ctx['facets_index'].keys())}")

🏗️  Building search context...
✅ Search context created!
   - BM25 models for fields: ['title', 'tags', 'desc']
   - TF-IDF matrix shape: (10, 31)
   - Document norms precomputed: 10
   - Popularity scores normalized: 10
   - Facets indexed: ['size', 'style', 'color']


## 3. Basic Search Examples

Let's test the search engine with various types of queries.

In [5]:
def display_results(query, results, max_results=5):
    """Helper function to display search results nicely."""
    print(f"\n🔍 Query: '{query}'")
    print("=" * 50)
    
    if not results:
        print("No results found.")
        return
    
    for i, (doc_id, score) in enumerate(results[:max_results]):
        doc = next(d for d in DOCS if d['id'] == doc_id)
        print(f"{i+1}. [{doc_id}] Score: {score:.3f}")
        print(f"   Title: {doc['title']}")
        print(f"   Tags: {doc['tags']}")
        print(f"   Facets: size={doc['size']}, style={doc['style']}, color={doc['color']}")
        print(f"   Popularity: {doc['popularity']}")
        print()

### 3.1 Keyword Search

In [6]:
# Test basic keyword search
results = search("resume", ctx, CONFIG, k=5)
display_results("resume", results)


🔍 Query: 'resume'
1. [d6] Score: 4.366
   Title: Professional CV resume pack
   Tags: ['cv', 'resume', 'professional']
   Facets: size=A4, style=modern, color=black
   Popularity: 80

2. [d1] Score: 4.145
   Title: Minimal A4 resume template
   Tags: ['resume', 'template', 'a4', 'minimal']
   Facets: size=A4, style=minimal, color=white
   Popularity: 120

3. [d3] Score: 0.058
   Title: Modern business presentation slides
   Tags: ['presentation', 'slides', 'modern']
   Facets: size=square, style=modern, color=blue
   Popularity: 200

4. [d4] Score: 0.042
   Title: Gold logo brandmark pack
   Tags: ['logo', 'brandmark', 'gold']
   Facets: size=square, style=modern, color=gold
   Popularity: 160

5. [d2] Score: 0.013
   Title: Vintage poster design A3
   Tags: ['poster', 'vintage', 'a3']
   Facets: size=A3, style=vintage, color=black
   Popularity: 90



### 3.2 Synonym Expansion

In [7]:
# Test synonym expansion: "cv" should expand to "resume"
results = search("cv", ctx, CONFIG, k=5)
display_results("cv", results)

print("\n💡 Note: 'cv' expands to 'resume' via synonym mapping!")


🔍 Query: 'cv'
1. [d6] Score: 9.632
   Title: Professional CV resume pack
   Tags: ['cv', 'resume', 'professional']
   Facets: size=A4, style=modern, color=black
   Popularity: 80

2. [d1] Score: 4.743
   Title: Minimal A4 resume template
   Tags: ['resume', 'template', 'a4', 'minimal']
   Facets: size=A4, style=minimal, color=white
   Popularity: 120

3. [d3] Score: 0.058
   Title: Modern business presentation slides
   Tags: ['presentation', 'slides', 'modern']
   Facets: size=square, style=modern, color=blue
   Popularity: 200

4. [d4] Score: 0.042
   Title: Gold logo brandmark pack
   Tags: ['logo', 'brandmark', 'gold']
   Facets: size=square, style=modern, color=gold
   Popularity: 160

5. [d2] Score: 0.013
   Title: Vintage poster design A3
   Tags: ['poster', 'vintage', 'a3']
   Facets: size=A3, style=vintage, color=black
   Popularity: 90


💡 Note: 'cv' expands to 'resume' via synonym mapping!


### 3.3 Facet-Based Boosting

In [8]:
# Test facet boosting: documents matching facets get score boosts
results = search("A4 minimal template", ctx, CONFIG, k=5)
display_results("A4 minimal template", results)

print("\n💡 Note: Documents with size='A4' and style='minimal' get boosted!")


🔍 Query: 'A4 minimal template'
1. [d1] Score: 4.591
   Title: Minimal A4 resume template
   Tags: ['resume', 'template', 'a4', 'minimal']
   Facets: size=A4, style=minimal, color=white
   Popularity: 120

2. [d5] Score: 3.814
   Title: Instagram post template minimal
   Tags: ['instagram', 'template', 'minimal']
   Facets: size=instagram, style=minimal, color=white
   Popularity: 300

3. [d7] Score: 3.599
   Title: Travel photo collage template
   Tags: ['photo', 'image', 'collage', 'template']
   Facets: size=square, style=minimal, color=white
   Popularity: 140

4. [d8] Score: 2.000
   Title: Business card minimal black
   Tags: ['card', 'business', 'minimal', 'black']
   Facets: size=A4, style=minimal, color=black
   Popularity: 60

5. [d9] Score: 1.021
   Title: Event flyer modern blue
   Tags: ['flyer', 'event', 'modern', 'blue']
   Facets: size=A4, style=modern, color=blue
   Popularity: 110


💡 Note: Documents with size='A4' and style='minimal' get boosted!


## 4. Interactive Search Interface

In [9]:
# Try different queries
test_queries = [
    "resume cv",
    "gold logo modern", 
    "instagram minimal white",
    "presentation slides business"
]

for query in test_queries:
    results = search(query, ctx, CONFIG, k=3)
    display_results(query, results, max_results=3)


🔍 Query: 'resume cv'
1. [d6] Score: 10.939
   Title: Professional CV resume pack
   Tags: ['cv', 'resume', 'professional']
   Facets: size=A4, style=modern, color=black
   Popularity: 80

2. [d1] Score: 5.979
   Title: Minimal A4 resume template
   Tags: ['resume', 'template', 'a4', 'minimal']
   Facets: size=A4, style=minimal, color=white
   Popularity: 120

3. [d3] Score: 0.058
   Title: Modern business presentation slides
   Tags: ['presentation', 'slides', 'modern']
   Facets: size=square, style=modern, color=blue
   Popularity: 200


🔍 Query: 'gold logo modern'
1. [d4] Score: 13.216
   Title: Gold logo brandmark pack
   Tags: ['logo', 'brandmark', 'gold']
   Facets: size=square, style=modern, color=gold
   Popularity: 160

2. [d3] Score: 1.058
   Title: Modern business presentation slides
   Tags: ['presentation', 'slides', 'modern']
   Facets: size=square, style=modern, color=blue
   Popularity: 200

3. [d6] Score: 1.008
   Title: Professional CV resume pack
   Tags: ['cv', 'res

## 5. Run Comprehensive Tests

In [10]:
# Final validation - run the built-in comprehensive tests
from search_engine import run_tests

print("🧪 RUNNING BUILT-IN COMPREHENSIVE TESTS")
print("=" * 50)
run_tests()

🧪 RUNNING BUILT-IN COMPREHENSIVE TESTS
Running comprehensive search engine tests...
✓ Test 1: Basic keyword search
✓ Test 2: Synonym expansion
✓ Test 3: Facet boosting
✓ Test 4: Empty query handling
✓ Test 5: MMR diversity
✓ Test 6: Edge cases
✓ Test 7: Deterministic results
All tests passed! ✅
